In [ ]:
!pip install torch scikit-learn tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

print('✅ Imports done!')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────
ROOT = '/content/drive/MyDrive/Hackenza_MUPlovers/'

# Two feature directories to combine
FEAT_DIR_1   = ROOT + 'cache/features_normalized_783(Renan)/'   # 1st dataset
FEAT_DIR_2   = ROOT + 'external2/features_normalized_783/'      # 2nd dataset

# Val manifest — must have file_id and label columns
# We'll auto-build this from both folders below
MODEL_PATH   = ROOT + 'runs/best_model.pt'
OUTPUT_PATH  = ROOT + 'predictions_combined.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Verify paths exist
for p in [FEAT_DIR_1, FEAT_DIR_2, MODEL_PATH]:
    print(f'  {os.path.exists(p)} → {p}')

In [ ]:
# ── Build combined manifest from both folders ──────────────────
# Each entry: (file_id, feature_path, label)
# Labels come from val_manifest_split.csv — if your files have a
# separate label CSV, update the paths below.

VAL_MANIFEST_1 = ROOT + 'data/val_manifest_split.csv'       # labels for folder 1
VAL_MANIFEST_2 = ROOT + 'external2/val_manifest_split.csv'  # labels for folder 2
# 👆 If both datasets share the same manifest, set both to the same file

rows = []

# Load folder 1
if os.path.exists(VAL_MANIFEST_1):
    df1 = pd.read_csv(VAL_MANIFEST_1)
    for _, row in df1.iterrows():
        fpath = os.path.join(FEAT_DIR_1, f"{row['file_id']}.npy")
        if os.path.exists(fpath):
            rows.append({'file_id': row['file_id'], 'feat_path': fpath, 'label': int(row['label'])})
    print(f'Folder 1: {len(rows)} files loaded')
else:
    # No manifest — load all files, labels unknown
    for f in os.listdir(FEAT_DIR_1):
        if f.endswith('.npy'):
            rows.append({'file_id': f.replace('.npy',''), 'feat_path': os.path.join(FEAT_DIR_1, f), 'label': -1})
    print(f'Folder 1: {len(rows)} files (no manifest — labels set to -1)')

n1 = len(rows)

# Load folder 2
if os.path.exists(VAL_MANIFEST_2):
    df2 = pd.read_csv(VAL_MANIFEST_2)
    for _, row in df2.iterrows():
        fpath = os.path.join(FEAT_DIR_2, f"{row['file_id']}.npy")
        if os.path.exists(fpath):
            rows.append({'file_id': row['file_id'], 'feat_path': fpath, 'label': int(row['label'])})
    print(f'Folder 2: {len(rows) - n1} files loaded')
else:
    for f in os.listdir(FEAT_DIR_2):
        if f.endswith('.npy'):
            rows.append({'file_id': f.replace('.npy',''), 'feat_path': os.path.join(FEAT_DIR_2, f), 'label': -1})
    print(f'Folder 2: {len(rows) - n1} files (no manifest — labels set to -1)')

combined_df = pd.DataFrame(rows)
print(f'\n✅ Total combined: {len(combined_df)} files')
print(f'   Label distribution: {combined_df["label"].value_counts().to_dict()}')
combined_df.head()

In [ ]:
# ── Dataset that reads from explicit paths ──────────────────────
class CombinedDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        X     = np.load(row['feat_path']).astype(np.float32)
        X     = torch.tensor(X)
        label = torch.tensor(float(row['label']), dtype=torch.float32)
        return X, label


def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = [s.shape[0] for s in sequences]
    padded  = pad_sequence(sequences, batch_first=True)  # [B, T_max, D]
    mask    = torch.zeros(padded.shape[:2])
    for i, l in enumerate(lengths):
        mask[i, :l] = 1
    return padded, mask, torch.stack(labels)


dataset = CombinedDataset(combined_df)
loader  = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# Infer D from first file
sample_D = np.load(combined_df.iloc[0]['feat_path']).shape[1]
print(f'✅ Dataset ready: {len(dataset)} files, feature dim D = {sample_D}')

In [ ]:
# ── Model definition (same as training notebook) ────────────────
class TemporalEncoder(nn.Module):
    def __init__(self, input_dim=783, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.output_dim = hidden_dim * 2

    def forward(self, x):
        out, _ = self.gru(x)
        return out


class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask):
        scores  = self.attn(x).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        return torch.sum(x * weights.unsqueeze(-1), dim=1)


class AccentModel(nn.Module):
    def __init__(self, input_dim=783):
        super().__init__()
        self.encoder    = TemporalEncoder(input_dim)
        self.pool       = AttentionPooling(self.encoder.output_dim)
        self.classifier = nn.Sequential(
            nn.Linear(self.encoder.output_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x, mask):
        seq    = self.encoder(x)
        pooled = self.pool(seq, mask)
        return self.classifier(pooled).squeeze(-1)


# Load with dynamic input_dim from actual data
model = AccentModel(input_dim=sample_D).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f'✅ Model loaded! input_dim={sample_D}')

In [ ]:
# ── Run inference ───────────────────────────────────────────────
all_preds  = []
all_probs  = []
all_labels = []

with torch.no_grad():
    for X, mask, y in tqdm(loader):
        X, mask = X.to(device), mask.to(device)

        logits = model(X, mask)
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs > 0.5).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().astype(int).tolist())

print('✅ Inference done!')

In [ ]:
# ── Save predictions.csv ────────────────────────────────────────
pred_df = combined_df.copy()
pred_df['prediction'] = all_preds
pred_df['confidence'] = [round(p, 4) for p in all_probs]
pred_df = pred_df[['file_id', 'label', 'prediction', 'confidence']]
pred_df.to_csv(OUTPUT_PATH, index=False)

print(f'✅ Saved → {OUTPUT_PATH}')
print(pred_df.to_string(index=False))

In [ ]:
# ── Accuracy breakdown ──────────────────────────────────────────
has_labels = pred_df['label'] != -1

if has_labels.sum() > 0:
    labeled = pred_df[has_labels]
    acc = accuracy_score(labeled['label'], labeled['prediction'])
    f1  = f1_score(labeled['label'], labeled['prediction'], zero_division=0)

    print(f'\n🏆 COMBINED RESULTS ({len(labeled)} labeled files)')
    print(f'   Accuracy : {acc:.4f} ({int(acc*len(labeled))}/{len(labeled)})')
    print(f'   F1 Score : {f1:.4f}')

    # Per-folder breakdown
    n1_ids = set(os.listdir(FEAT_DIR_1))
    folder1 = labeled[labeled['file_id'].astype(str).apply(lambda x: f'{x}.npy').isin(n1_ids)]
    folder2 = labeled[~labeled['file_id'].astype(str).apply(lambda x: f'{x}.npy').isin(n1_ids)]

    if len(folder1) > 0:
        acc1 = accuracy_score(folder1['label'], folder1['prediction'])
        print(f'\n   Folder 1 (Renan)    : {acc1:.4f} ({len(folder1)} files)')
    if len(folder2) > 0:
        acc2 = accuracy_score(folder2['label'], folder2['prediction'])
        print(f'   Folder 2 (external2): {acc2:.4f} ({len(folder2)} files)')
else:
    print('⚠️  No labels found — predictions saved but accuracy cannot be computed')